# Analysis outline

See [README.md](README.md).

# Set up

Run the following cell.  It loads some required functions.

In [36]:
run kondrashov

# Search for human sequences

Below we try to retrieve human sequences for each gene programatically.  We
choose the [RefSeq Select](https://www.ncbi.nlm.nih.gov/refseq/refseq_select/) database.

In [38]:
# genes in Kondrashov et al. (2002)
print(loci)

['ABCD1', 'ALPL', 'AR', 'ATP7B', 'BTK', 'CASR', 'CBS', 'CFTR', 'CYBB', 'F7', 'F8', 'F9', 'G6PD', 'GALT', 'GBA', 'GJB1', 'HBB', 'HPRT1', 'IL2RG', 'KCNH2', 'KCNQ1', 'L1CAM', 'LDLR', 'MPZ', 'MYH7', 'TYR', 'PAH', 'PMM2', 'RHO', 'TP53', 'TTR', 'VWF']


In [66]:
genes = {}
for locus in loci:
    query = f"Homo sapiens[Organism] AND {locus}[Gene Name] AND refseq_select[Filter]"
    handle = Entrez.esearch(db="nucleotide", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
    genes.update({locus: record['IdList']})

ABCD1: 1 -> ['1519313321']
ALPL: 1 -> ['1519315936']
AR: 3 -> ['1654124212', '1519245710', '1519243240']
ATP7B: 1 -> ['1677498587']
BTK: 1 -> ['1780222528']
CASR: 1 -> ['1677537479']
CBS: 1 -> ['1780222567']
CFTR: 1 -> ['1732746288']
CYBB: 1 -> ['1732746192']
F7: 1 -> ['1779521762']
F8: 1 -> ['1812229661']
F9: 1 -> ['1732746198']
G6PD: 1 -> ['1497290737']
GALT: 1 -> ['1677502190']
GBA: 1 -> ['1519244100']
GJB1: 1 -> ['1606717073']
HBB: 1 -> ['1401724401']
HPRT1: 1 -> ['1519243265']
IL2RG: 1 -> ['1780222514']
KCNH2: 1 -> ['1732746325']
KCNQ1: 1 -> ['1732746161']
L1CAM: 1 -> ['1653961419']
LDLR: 1 -> ['1732746181']
MPZ: 1 -> ['1519245315']
MYH7: 1 -> ['1519242569']
TYR: 1 -> ['1519243869']
PAH: 1 -> ['1519244862']
PMM2: 1 -> ['1519243247']
RHO: 2 -> ['1519316345', '169808383']
TP53: 1 -> ['1808862652']
TTR: 1 -> ['1780222569']
VWF: 1 -> ['1813372008']


Two of the genes, AR and RHO, have more than one RefSeq Select transcripts.
Let's look at them more closely.

* AR: only the first sequence is valid.

* RHO: only the second sequence is valid.

In [69]:
# AR
for i in genes['AR']:
    stream = Entrez.efetch(db="nucleotide", id=i, rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    print(f"ID: {i}")
    print(f"{record2.id}: {record2.description}\nLength: {len(record2.seq)} nt\n")

ID: 1654124212
NM_000044.6: Homo sapiens androgen receptor (AR), transcript variant 1, mRNA
Length: 10667 nt

ID: 1519245710
NM_001657.4: Homo sapiens amphiregulin (AREG), mRNA
Length: 1234 nt

ID: 1519243240
NM_001628.4: Homo sapiens aldo-keto reductase family 1 member B (AKR1B1), transcript variant 1, mRNA
Length: 1361 nt



In [71]:
del genes['AR'][1]
del genes['AR'][1]
genes['AR']

['1654124212']

In [70]:
# RHO
for i in genes['RHO']:
    stream = Entrez.efetch(db="nucleotide", id=i, rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    print(f"ID: {i}")
    print(f"{record2.id}: {record2.description}\nLength: {len(record2.seq)} nt\n")

ID: 1519316345
NM_014578.4: Homo sapiens ras homolog family member D (RHOD), transcript variant 1, mRNA
Length: 1104 nt

ID: 169808383
NM_000539.3: Homo sapiens rhodopsin (RHO), mRNA
Length: 2768 nt



In [72]:
del genes['RHO'][0]
genes['RHO']

['169808383']

In [77]:
for gene in genes:
    stream = Entrez.efetch(db="nucleotide", id=genes[gene][0], rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    genes[gene].append(record2.id)
    print(gene, genes[gene])

ABCD1 ['1519313321', 'NM_000033.4']
ALPL ['1519315936', 'NM_000478.6']
AR ['1654124212', 'NM_000044.6']
ATP7B ['1677498587', 'NM_000053.4']
BTK ['1780222528', 'NM_000061.3']
CASR ['1677537479', 'NM_000388.4']
CBS ['1780222567', 'NM_000071.3']
CFTR ['1732746288', 'NM_000492.4']
CYBB ['1732746192', 'NM_000397.4']
F7 ['1779521762', 'NM_019616.4']
F8 ['1812229661', 'NM_000132.4']
F9 ['1732746198', 'NM_000133.4']
G6PD ['1497290737', 'NM_001360016.2']
GALT ['1677502190', 'NM_000155.4']
GBA ['1519244100', 'NM_000157.4']
GJB1 ['1606717073', 'NM_000166.6']
HBB ['1401724401', 'NM_000518.5']
HPRT1 ['1519243265', 'NM_000194.3']
IL2RG ['1780222514', 'NM_000206.3']
KCNH2 ['1732746325', 'NM_000238.4']
KCNQ1 ['1732746161', 'NM_000218.3']
L1CAM ['1653961419', 'NM_001278116.2']
LDLR ['1732746181', 'NM_000527.5']
MPZ ['1519245315', 'NM_000530.8']
MYH7 ['1519242569', 'NM_000257.4']
TYR ['1519243869', 'NM_000372.5']
PAH ['1519244862', 'NM_000277.3']
PMM2 ['1519243247', 'NM_000303.3']
RHO ['169808383', 'NM_

# Search for orthologs

Primates are represented by [Registry Number txid9443](https://www.ncbi.nlm.nih.gov/mesh/68011323).

In [65]:
for locus in loci:
    query = f"{locus}[Gene Name] AND ortholog AND txid9443[Organism]"
    handle = Entrez.esearch(db="nucleotide", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])

ABCD1: 19 -> ['2671706189', '2671508533', '2671424538', '2976869235']
ALPL: 25 -> ['2671706197', '2671508593', '2671424629', '2976869251']
AR: 21 -> ['2671706189', '2671508533', '2671424538', '2976869235']
ATP7B: 4 -> ['2976869257', '808395705', '786102498', '778163266']
BTK: 20 -> ['2671706189', '2671508533', '2671424538', '2976869235']
CASR: 19 -> ['2671508590', '2671424624', '2976869243', '3008884616']
CBS: 11 -> ['3008884616', '2724200167', '2946946711', '2811273532']
CFTR: 24 -> ['2671706238', '2671508578', '2671424607', '2976869247']
CYBB: 10 -> ['2671706189', '2671508533', '2671424538', '2976869235']
F7: 1 -> ['2976869257']
F8: 20 -> ['2671706189', '2671508533', '2671424538', '2976869235']
F9: 22 -> ['2671706189', '2671508533', '2671424538', '2976869235']
G6PD: 17 -> ['2671706189', '2671508533', '2671424538', '2976869235']
GALT: 31 -> ['2671508560', '2671424581', '2976869257', '3047139651']
GBA: 9 -> ['2671424629', '2131398679', '1823692222', '1759440049']
GJB1: 20 -> ['26717061